# 从零实现 GAT：边注意力、目标分段 Softmax 与多头网络

本 Notebook 只用 PyTorch 基础张量和 `nn.Module` 手写 edge attention、按目标节点的 segment softmax、`GATLayer` 与两层多头 `GATNet`；不使用 PyG、DGL、`torch_geometric` 或现成 GNN 层。

链路包括 tenant 前置过滤、邻接 mask、自环去重、数值稳定 softmax、参数量/shape、mask 半监督训练、梯度、注意力和为 1、标签与 tenant 泄漏反例、state/artifact 指纹和推理合同。数据完全虚构、CPU 离线且固定种子。

In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import time  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED=2701  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE=torch.device("cpu")  # 计算并保存当前步骤的中间状态。
DTYPE=torch.float32  # 计算并保存当前步骤的中间状态。

def canonical_fingerprint(payload)->str:  # 定义本节可复用的核心函数。
    raw=json.dumps(payload,ensure_ascii=False,sort_keys=True,separators=(",",":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]  # 返回当前分支计算出的结果。

assert DEVICE.type=="cpu" and torch.get_num_threads()==1  # 用受控断言验证关键不变量。
assert torch.initial_seed()==SEED  # 用受控断言验证关键不变量。
assert not any(name in globals() for name in ("torch_geometric","dgl"))  # 用受控断言验证关键不变量。

## 1. 受控图与 tenant 合同

tenant-a 有 18 个节点、两类各 9 个；同类环和二跳边占主导，另有少量跨类边。tenant-b 的极端特征以及一条恶意跨 tenant 边用于验证隔离。注意力不是先在全图计算再隐藏结果：softmax 的分母会因不可见邻居改变，因此 tenant/ACL 必须在 edge mask 和自环生成之前完成。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AuthContext:  # 定义承载本节状态与行为的数据结构。
    tenant:str  # 执行当前语句以推进本节示例。
    scopes:frozenset[str]  # 执行当前语句以推进本节示例。
    principal:str  # 执行当前语句以推进本节示例。
    def require(self,scope:str)->None:  # 定义本节可复用的核心函数。
        if scope not in self.scopes:  # 按当前条件选择后续控制路径。
            raise PermissionError(f"缺少 scope: {scope}")  # 遇到非法合同立即显式失败。

auth_a=AuthContext("tenant-a",frozenset({"graph:read","model:predict"}),"alice")  # 计算并保存当前步骤的中间状态。
node_ids_all=[f"tenant-a:n{i:02d}" for i in range(18)]+["tenant-b:x0","tenant-b:x1"]  # 计算并保存当前步骤的中间状态。
tenant_all=["tenant-a"]*18+["tenant-b"]*2  # 计算并保存当前步骤的中间状态。
generator=torch.Generator().manual_seed(SEED)  # 计算并保存当前步骤的中间状态。
base0,base1=torch.tensor([1.15,0.2,0.35,1.0]),torch.tensor([0.2,1.15,0.75,1.0])  # 计算并保存当前步骤的中间状态。
X_a=torch.stack([(base0 if i<9 else base1)+0.23*torch.randn(4,generator=generator)*torch.tensor([1,1,1,0]) for i in range(18)])  # 计算并保存当前步骤的中间状态。
X_all=torch.cat([X_a,torch.tensor([[100.,100.,100.,1.],[-100.,-100.,-100.,1.]])])  # 计算并保存当前步骤的中间状态。
y_all=torch.tensor([0]*9+[1]*9+[-1,-1],dtype=torch.long)  # 计算并保存当前步骤的中间状态。

pair_set=set()  # 计算并保存当前步骤的中间状态。
for offset in (0,9):  # 遍历输入元素以累积或检查结果。
    for j in range(9):  # 遍历输入元素以累积或检查结果。
        for step in (1,2):  # 遍历输入元素以累积或检查结果。
            pair_set.add(tuple(sorted((offset+j,offset+(j+step)%9))))  # 执行当前语句以推进本节示例。
raw_pairs=sorted(pair_set|{(0,9),(4,13),(18,19),(0,18)})  # 计算并保存当前步骤的中间状态。

def authorize(auth:AuthContext):  # 定义本节可复用的核心函数。
    auth.require("graph:read")  # 执行当前语句以推进本节示例。
    visible=[i for i,t in enumerate(tenant_all) if t==auth.tenant]  # 计算并保存当前步骤的中间状态。
    remap={old:new for new,old in enumerate(visible)}  # 计算并保存当前步骤的中间状态。
    pairs=[(remap[u],remap[v]) for u,v in raw_pairs if u in remap and v in remap and tenant_all[u]==tenant_all[v]==auth.tenant]  # 计算并保存当前步骤的中间状态。
    return [node_ids_all[i] for i in visible],X_all[visible].clone(),y_all[visible].clone(),sorted(pairs)  # 返回当前分支计算出的结果。

node_ids,X,y,undirected_pairs=authorize(auth_a)  # 计算并保存当前步骤的中间状态。
assert X.shape==(18,4) and y.shape==(18,)  # 用受控断言验证关键不变量。
assert len(undirected_pairs)==38  # 用受控断言验证关键不变量。
assert set(y.tolist())=={0,1}  # 用受控断言验证关键不变量。
assert all(n.startswith("tenant-a:") for n in node_ids)  # 用受控断言验证关键不变量。

## 2. Mask 与半监督边界

每类 4 个 train、2 个 validation、3 个 test。所有授权节点的无标签特征与边参与 transductive attention，只有 train 标签进入交叉熵；validation 选择 checkpoint，test 冻结后只打开一次。GAT 可用于 inductive 图，但本 Notebook 的基础评估明确是 transductive，不能因模型名称而改写协议。

In [ ]:
train_mask=torch.zeros(18,dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
val_mask=torch.zeros(18,dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
test_mask=torch.zeros(18,dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
for cls in (0,1):  # 遍历输入元素以累积或检查结果。
    idx=torch.where(y==cls)[0]  # 计算并保存当前步骤的中间状态。
    train_mask[idx[:4]]=True; val_mask[idx[4:6]]=True; test_mask[idx[6:]]=True  # 计算并保存当前步骤的中间状态。
coverage=train_mask.to(torch.int8)+val_mask.to(torch.int8)+test_mask.to(torch.int8)  # 计算并保存当前步骤的中间状态。
assert (int(train_mask.sum()),int(val_mask.sum()),int(test_mask.sum()))==(8,4,6)  # 用受控断言验证关键不变量。
assert torch.equal(coverage,torch.ones_like(coverage))  # 用受控断言验证关键不变量。
assert set(y[train_mask].tolist())==set(y[val_mask].tolist())==set(y[test_mask].tolist())=={0,1}  # 用受控断言验证关键不变量。
assert not torch.any(train_mask&val_mask) and not torch.any(train_mask&test_mask)  # 用受控断言验证关键不变量。

## 3. 邻接 mask 与自环

约定 `source→target`：target 从 source 接收消息。无向边展开为两个方向；`with_exact_self_loops` 先去重全部边，再为每个节点恰好加入一个自环。GAT 只能在这些边上算 attention，不能为非邻接节点创建一个大负数后忘记彻底 mask。自环确保每个目标至少有一个入边，softmax 分母非空。

In [ ]:
def directed_edges(num_nodes:int,pairs:list[tuple[int,int]])->torch.Tensor:  # 定义本节可复用的核心函数。
    directed=[]  # 计算并保存当前步骤的中间状态。
    for u,v in pairs:  # 遍历输入元素以累积或检查结果。
        if u==v or not (0<=u<num_nodes and 0<=v<num_nodes):  # 按当前条件选择后续控制路径。
            raise ValueError("边端点非法或输入含自环")  # 遇到非法合同立即显式失败。
        directed.extend([(u,v),(v,u)])  # 执行当前语句以推进本节示例。
    directed=sorted(set(directed),key=lambda p:(p[1],p[0]))  # 计算并保存当前步骤的中间状态。
    return torch.tensor(directed,dtype=torch.long).T.contiguous()  # 返回当前分支计算出的结果。

def with_exact_self_loops(edges:torch.Tensor,num_nodes:int)->torch.Tensor:  # 定义本节可复用的核心函数。
    if edges.ndim!=2 or edges.shape[0]!=2:  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index shape 必须为 (2,E)")  # 遇到非法合同立即显式失败。
    pairs={(int(s),int(t)) for s,t in edges.T.tolist() if int(s)!=int(t)}  # 计算并保存当前步骤的中间状态。
    if any(s<0 or t<0 or s>=num_nodes or t>=num_nodes for s,t in pairs):  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 越界")  # 遇到非法合同立即显式失败。
    pairs.update((i,i) for i in range(num_nodes))  # 执行当前语句以推进本节示例。
    ordered=sorted(pairs,key=lambda p:(p[1],p[0]))  # 计算并保存当前步骤的中间状态。
    return torch.tensor(ordered,dtype=torch.long).T.contiguous()  # 返回当前分支计算出的结果。

base_edges=directed_edges(18,undirected_pairs)  # 计算并保存当前步骤的中间状态。
masked_edges=with_exact_self_loops(base_edges,18)  # 计算并保存当前步骤的中间状态。
loop_mask=masked_edges[0]==masked_edges[1]  # 计算并保存当前步骤的中间状态。
loops_per_target=torch.bincount(masked_edges[1,loop_mask],minlength=18)  # 计算并保存当前步骤的中间状态。
assert base_edges.shape==(2,76) and masked_edges.shape==(2,94)  # 用受控断言验证关键不变量。
assert torch.equal(loops_per_target,torch.ones(18,dtype=torch.long))  # 用受控断言验证关键不变量。
assert (0,5) not in set(map(tuple,masked_edges.T.tolist()))  # 用受控断言验证关键不变量。
assert all(int((masked_edges[1]==i).sum())>0 for i in range(18))  # 用受控断言验证关键不变量。

## 4. 按目标节点的 segment softmax

每条边、每个 head 有一个 logit (e_{s\to t}^{(h)})。归一化必须只在相同 target 的入边集合中进行：

[
\alpha_{s\to t}^{(h)}=
\frac{\exp(e_{s\to t}^{(h)}-m_t^{(h)})}
{\sum_{k\in N(t)\cup\{t\}}\exp(e_{k\to t}^{(h)}-m_t^{(h)})}
]

减去 segment 最大值避免溢出。对每个存在入边的 target/head，attention 之和应为 1。

In [ ]:
def target_segment_softmax(scores:torch.Tensor,target:torch.Tensor,num_nodes:int)->torch.Tensor:  # 定义本节可复用的核心函数。
    if scores.ndim!=2 or target.ndim!=1 or scores.shape[0]!=target.numel():  # 按当前条件选择后续控制路径。
        raise ValueError("scores/target shape 不匹配")  # 遇到非法合同立即显式失败。
    if target.numel() and (int(target.min())<0 or int(target.max())>=num_nodes):  # 按当前条件选择后续控制路径。
        raise ValueError("target 越界")  # 遇到非法合同立即显式失败。
    alpha=torch.zeros_like(scores)  # 计算并保存当前步骤的中间状态。
    for node in range(num_nodes):  # 遍历输入元素以累积或检查结果。
        mask=target==node  # 计算并保存当前步骤的中间状态。
        if mask.any():  # 按当前条件选择后续控制路径。
            local=scores[mask]  # 计算并保存当前步骤的中间状态。
            shifted=local-local.max(dim=0,keepdim=True).values  # 计算并保存当前步骤的中间状态。
            exp=shifted.exp()  # 计算并保存当前步骤的中间状态。
            alpha[mask]=exp/exp.sum(dim=0,keepdim=True)  # 计算并保存当前步骤的中间状态。
    return alpha  # 返回当前分支计算出的结果。

fixture_scores=torch.tensor([[1000.,-2.],[999.,-1.],[4.,7.],[4.,5.]])  # 计算并保存当前步骤的中间状态。
fixture_target=torch.tensor([0,0,1,1])  # 计算并保存当前步骤的中间状态。
fixture_alpha=target_segment_softmax(fixture_scores,fixture_target,2)  # 计算并保存当前步骤的中间状态。
for node in (0,1):  # 遍历输入元素以累积或检查结果。
    assert torch.allclose(fixture_alpha[fixture_target==node].sum(0),torch.ones(2),atol=1e-7)  # 用受控断言验证关键不变量。
np_ref=np.exp(np.array([1000.,999.])-1000.); np_ref=np_ref/np_ref.sum()  # 计算并保存当前步骤的中间状态。
assert np.allclose(fixture_alpha[:2,0].numpy(),np_ref,atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.isfinite(fixture_alpha).all() and torch.all(fixture_alpha>0)  # 用受控断言验证关键不变量。

## 5. 手写多头 `GATLayer`

每个 head 有 (W^{(h)}\in\mathbb{R}^{F_{in}\times F_{out}})、(a_s^{(h)},a_t^{(h)}\in\mathbb{R}^{F_{out}})。变换后：

[
e_{s\to t}^{(h)}=\operatorname{LeakyReLU}
((a_s^{(h)})^TWh_s+(a_t^{(h)})^TWh_t)
]

随后 target-segment softmax，并按 target 累加 (alpha Wh_s)。concat 模式输出 `(N,H*F_out)`，mean 模式输出 `(N,F_out)`。

主数据图把无向边展开成双向边，单靠它无法充分锁定 `source→target` 落点；而第二层只有一个 head，`concat=False` 也退化为恒等。下面增加固定权重的非对称 `0→1` fixture，并令两个 head 使用 1×/3× 投影。预期输出可精确手算，同时验证消息方向和真正的多头 mean。


In [ ]:
class GATLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,in_features:int,out_features:int,heads:int=1,concat:bool=True,  # 定义本节可复用的核心函数。
                 negative_slope:float=0.2,attention_dropout:float=0.0):  # 计算并保存当前步骤的中间状态。
        super().__init__()  # 执行当前语句以推进本节示例。
        if min(in_features,out_features,heads)<=0:  # 按当前条件选择后续控制路径。
            raise ValueError("维度与 heads 必须为正")  # 遇到非法合同立即显式失败。
        self.in_features,self.out_features,self.heads=in_features,out_features,heads  # 计算并保存当前步骤的中间状态。
        self.concat,self.negative_slope=bool(concat),float(negative_slope)  # 计算并保存当前步骤的中间状态。
        self.attention_dropout=float(attention_dropout)  # 计算并保存当前步骤的中间状态。
        self.weight=nn.Parameter(torch.empty(heads,in_features,out_features))  # 计算并保存当前步骤的中间状态。
        self.attn_source=nn.Parameter(torch.empty(heads,out_features))  # 计算并保存当前步骤的中间状态。
        self.attn_target=nn.Parameter(torch.empty(heads,out_features))  # 计算并保存当前步骤的中间状态。
        bias_size=heads*out_features if concat else out_features  # 计算并保存当前步骤的中间状态。
        self.bias=nn.Parameter(torch.zeros(bias_size))  # 计算并保存当前步骤的中间状态。
        nn.init.xavier_uniform_(self.weight)  # 执行当前语句以推进本节示例。
        nn.init.xavier_uniform_(self.attn_source.unsqueeze(-1))  # 执行当前语句以推进本节示例。
        nn.init.xavier_uniform_(self.attn_target.unsqueeze(-1))  # 执行当前语句以推进本节示例。

    def forward(self,x:torch.Tensor,edges:torch.Tensor,return_attention:bool=False):  # 定义本节可复用的核心函数。
        if x.ndim!=2 or x.shape[1]!=self.in_features:  # 按当前条件选择后续控制路径。
            raise ValueError("x shape 不匹配")  # 遇到非法合同立即显式失败。
        used_edges=with_exact_self_loops(edges,x.shape[0]).to(x.device)  # 计算并保存当前步骤的中间状态。
        source,target=used_edges  # 计算并保存当前步骤的中间状态。
        transformed=torch.einsum("nf,hfo->nho",x,self.weight)  # 计算并保存当前步骤的中间状态。
        score_source=(transformed[source]*self.attn_source.unsqueeze(0)).sum(-1)  # 计算并保存当前步骤的中间状态。
        score_target=(transformed[target]*self.attn_target.unsqueeze(0)).sum(-1)  # 计算并保存当前步骤的中间状态。
        scores=F.leaky_relu(score_source+score_target,negative_slope=self.negative_slope)  # 计算并保存当前步骤的中间状态。
        alpha=target_segment_softmax(scores,target,x.shape[0])  # 计算并保存当前步骤的中间状态。
        message_alpha=F.dropout(alpha,p=self.attention_dropout,training=self.training)  # 计算并保存当前步骤的中间状态。
        messages=message_alpha.unsqueeze(-1)*transformed[source]  # 计算并保存当前步骤的中间状态。
        aggregated=torch.zeros((x.shape[0],self.heads,self.out_features),dtype=x.dtype,device=x.device)  # 计算并保存当前步骤的中间状态。
        aggregated.index_add_(0,target,messages)  # 执行当前语句以推进本节示例。
        out=aggregated.reshape(x.shape[0],-1) if self.concat else aggregated.mean(dim=1)  # 计算并保存当前步骤的中间状态。
        out=out+self.bias  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(out).all():  # 按当前条件选择后续控制路径。
            raise ValueError("GATLayer 产生非有限值")  # 遇到非法合同立即显式失败。
        return (out,alpha,used_edges) if return_attention else out  # 返回当前分支计算出的结果。

probe_layer=GATLayer(4,3,heads=2,concat=True)  # 计算并保存当前步骤的中间状态。
probe,probe_alpha,probe_edges=probe_layer(X,base_edges,return_attention=True)  # 计算并保存当前步骤的中间状态。
assert probe.shape==(18,6) and probe_alpha.shape==(94,2)  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in probe_layer.parameters())==2*4*3+2*2*3+2*3  # 用受控断言验证关键不变量。
assert probe_edges.shape==masked_edges.shape  # 用受控断言验证关键不变量。
assert torch.isfinite(probe).all()  # 用受控断言验证关键不变量。
# 非对称有向 oracle：只有 0→1（另由层内部恰好加入 0→0、1→1）。
oracle_layer27 = GATLayer(1, 1, heads=2, concat=False, attention_dropout=0.0)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    oracle_layer27.weight.zero_()  # 执行当前语句以推进本节示例。
    oracle_layer27.weight[0, 0, 0] = 1.0  # 计算并保存当前步骤的中间状态。
    oracle_layer27.weight[1, 0, 0] = 3.0  # 计算并保存当前步骤的中间状态。
    oracle_layer27.attn_source.zero_(); oracle_layer27.attn_target.zero_(); oracle_layer27.bias.zero_()  # 执行当前语句以推进本节示例。
oracle_layer27.eval()  # 执行当前语句以推进本节示例。
oracle_X27 = torch.tensor([[2.0], [10.0]])  # 计算并保存当前步骤的中间状态。
oracle_edges27 = torch.tensor([[0], [1]], dtype=torch.long)  # source row, target row；中文说明：该行遵循既定约束。
oracle_out27, oracle_alpha27, oracle_used27 = oracle_layer27(  # 计算并保存当前步骤的中间状态。
    oracle_X27, oracle_edges27, return_attention=True  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
oracle_edge_set27 = set(map(tuple, oracle_used27.T.tolist()))  # 计算并保存当前步骤的中间状态。
assert oracle_edge_set27 == {(0, 0), (0, 1), (1, 1)}  # 用受控断言验证关键不变量。
assert (1, 0) not in oracle_edge_set27  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_out27, torch.tensor([[4.0], [12.0]]), atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_alpha27[oracle_used27[1] == 0], torch.ones(1, 2), atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_alpha27[oracle_used27[1] == 1], torch.full((2, 2), 0.5), atol=1e-7)  # 用受控断言验证关键不变量。
assert oracle_layer27.heads == 2 and oracle_layer27.concat is False  # 用受控断言验证关键不变量。


## 6. 两层 `GATNet` 与参数量

第一层 2 个 head、每头 4 维并 concat，得到 8 维；ELU/Dropout 后，第二层 1 个 head 输出 2 类且不 concat。第一层参数 56，第二层 22，总计 78。返回的是 logits；attention 只在诊断模式返回，避免服务默认泄露完整邻接证据。

In [ ]:
class GATNet(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,in_features:int,hidden_per_head:int,classes:int,heads:int=2,dropout:float=0.1):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.gat1=GATLayer(in_features,hidden_per_head,heads=heads,concat=True,attention_dropout=0.05)  # 计算并保存当前步骤的中间状态。
        self.gat2=GATLayer(hidden_per_head*heads,classes,heads=1,concat=False,attention_dropout=0.0)  # 计算并保存当前步骤的中间状态。
        self.dropout=float(dropout)  # 计算并保存当前步骤的中间状态。

    def forward(self,x:torch.Tensor,edges:torch.Tensor,return_attention:bool=False):  # 定义本节可复用的核心函数。
        if return_attention:  # 按当前条件选择后续控制路径。
            h,a1,e1=self.gat1(x,edges,return_attention=True)  # 计算并保存当前步骤的中间状态。
            h=F.elu(h); h=F.dropout(h,p=self.dropout,training=self.training)  # 计算并保存当前步骤的中间状态。
            logits,a2,e2=self.gat2(h,edges,return_attention=True)  # 计算并保存当前步骤的中间状态。
            return logits,{"layer1":(a1,e1),"layer2":(a2,e2)}  # 返回当前分支计算出的结果。
        h=F.elu(self.gat1(x,edges))  # 计算并保存当前步骤的中间状态。
        h=F.dropout(h,p=self.dropout,training=self.training)  # 计算并保存当前步骤的中间状态。
        return self.gat2(h,edges)  # 返回当前分支计算出的结果。

model=GATNet(4,4,2,heads=2)  # 计算并保存当前步骤的中间状态。
model.eval(); logits,attention=model(X,base_edges,return_attention=True)  # 计算并保存当前步骤的中间状态。
assert logits.shape==(18,2)  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in model.parameters())==78  # 用受控断言验证关键不变量。
assert attention["layer1"][0].shape==(94,2) and attention["layer2"][0].shape==(94,1)  # 用受控断言验证关键不变量。
assert torch.isfinite(logits).all()  # 用受控断言验证关键不变量。

## 7. Attention 不变量：每个 target/head 和为 1

该断言必须按 target 分组；对全图或 source 求和都不是 GAT 的归一化语义。还要确认 alpha 只对应 mask 后的边，自环恰好一次。训练态的 attention dropout 会让实际消息权重不再严格和为 1，因此此处在 `eval()` 下检查返回的 dropout 前 alpha。

In [ ]:
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _,attention=model(X,base_edges,return_attention=True)  # 计算并保存当前步骤的中间状态。
for layer_name,(alpha,edges_used) in attention.items():  # 遍历输入元素以累积或检查结果。
    targets=edges_used[1]  # 计算并保存当前步骤的中间状态。
    for node in range(18):  # 遍历输入元素以累积或检查结果。
        sums=alpha[targets==node].sum(dim=0)  # 计算并保存当前步骤的中间状态。
        assert torch.allclose(sums,torch.ones_like(sums),atol=1e-6), (layer_name,node,sums)  # 用受控断言验证关键不变量。
    assert torch.all(alpha>=0) and torch.all(alpha<=1)  # 用受控断言验证关键不变量。
    assert torch.equal(torch.bincount(edges_used[1,edges_used[0]==edges_used[1]],minlength=18),torch.ones(18,dtype=torch.long))  # 用受控断言验证关键不变量。
assert set(map(tuple,attention["layer1"][1].T.tolist()))==set(map(tuple,masked_edges.T.tolist()))  # 用受控断言验证关键不变量。
assert (0,5) not in set(map(tuple,attention["layer1"][1].T.tolist()))  # 用受控断言验证关键不变量。

## 8. Train mask 训练与 validation checkpoint

每轮 full-batch forward 只在授权边上做 attention；交叉熵严格索引 `train_mask`。validation loss 只选择 checkpoint，不反传。保存 detached clone，恢复后才计算 test。小图上的 full-batch Python segment 循环用于清晰性，生产不能据此估算吞吐。

训练前保存参数副本与 eval-mode train loss；恢复最佳 checkpoint 后要求参数确实变化、train loss 明显下降、train-mask accuracy 达标。这样 `best_state`、有限 history 和非零梯度不再掩盖误删 `optimizer.step()` 的回归。


In [ ]:
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
model = GATNet(4, 4, 2, heads=2, dropout=0.1)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(model.parameters(), lr=0.035, weight_decay=5e-4)  # 计算并保存当前步骤的中间状态。
initial_train_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # 计算并保存当前步骤的中间状态。
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_train_eval_loss = float(F.cross_entropy(model(X, base_edges)[train_mask], y[train_mask]))  # 计算并保存当前步骤的中间状态。

best_val, best_epoch, best_state = math.inf, -1, None  # 计算并保存当前步骤的中间状态。
history = []; started = time.perf_counter()  # 计算并保存当前步骤的中间状态。
for epoch in range(120):  # 遍历输入元素以累积或检查结果。
    model.train(); optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits = model(X, base_edges)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits[train_mask], y[train_mask])  # 计算并保存当前步骤的中间状态。
    loss.backward(); optimizer.step()  # 执行当前语句以推进本节示例。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        val_loss = float(F.cross_entropy(model(X, base_edges)[val_mask], y[val_mask]))  # 计算并保存当前步骤的中间状态。
    history.append((float(loss.detach()), val_loss))  # 执行当前语句以推进本节示例。
    if val_loss < best_val:  # 按当前条件选择后续控制路径。
        best_val, best_epoch = val_loss, epoch  # 计算并保存当前步骤的中间状态。
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # 计算并保存当前步骤的中间状态。
train_seconds = time.perf_counter() - started  # 计算并保存当前步骤的中间状态。

assert best_state is not None and 0 <= best_epoch < 120  # 用受控断言验证关键不变量。
assert len(history) == 120 and np.isfinite(np.asarray(history)).all()  # 用受控断言验证关键不变量。
assert train_seconds < 20  # 用受控断言验证关键不变量。
model.load_state_dict(best_state, strict=True); model.eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    restored_train_logits = model(X, base_edges)  # 计算并保存当前步骤的中间状态。
    restored_train_eval_loss = float(F.cross_entropy(restored_train_logits[train_mask], y[train_mask]))  # 计算并保存当前步骤的中间状态。
    restored_train_accuracy = float((restored_train_logits[train_mask].argmax(1) == y[train_mask]).float().mean())  # 计算并保存当前步骤的中间状态。
updated_parameter_keys = [key for key, value in model.state_dict().items()  # 计算并保存当前步骤的中间状态。
                          if not torch.equal(value.detach(), initial_train_state[key])]  # 按当前条件选择后续控制路径。
assert updated_parameter_keys, "optimizer 没有改变任何 state_dict 张量"  # 用受控断言验证关键不变量。
assert restored_train_eval_loss < initial_train_eval_loss * 0.75  # 用受控断言验证关键不变量。
assert restored_train_accuracy >= 0.90  # 用受控断言验证关键不变量。


## 9. 非零有限梯度

GAT 常见错误是把 alpha 转 NumPy、在 segment 操作中 detach，或用不可微覆盖导致 attention 参数无梯度。下面对恢复后的模型做一次 train-mask backward，要求权重、source attention、target attention和 bias 均有有限非零梯度；不执行 optimizer step，因此不会改变制品。

In [ ]:
model.train(); optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
check_loss=F.cross_entropy(model(X,base_edges)[train_mask],y[train_mask])  # 计算并保存当前步骤的中间状态。
check_loss.backward()  # 执行当前语句以推进本节示例。
grad_norms={name:float(p.grad.norm()) for name,p in model.named_parameters()}  # 计算并保存当前步骤的中间状态。
assert len(grad_norms)==8  # 用受控断言验证关键不变量。
assert all(math.isfinite(v) and v>0 for v in grad_norms.values())  # 用受控断言验证关键不变量。
assert any("attn_source" in name for name in grad_norms)  # 用受控断言验证关键不变量。
assert any("attn_target" in name for name in grad_norms)  # 用受控断言验证关键不变量。
model.eval()  # 执行当前语句以推进本节示例。

## 10. Test 与标签泄漏反例

冻结 checkpoint 后只报告一次 test accuracy。为了验证损失边界，把所有 test 标签翻转；只要训练损失确实只索引 train mask，数值必须完全不变。这个反例只验证标签索引，不替代特征 event-time、预处理 fit split 与 lineage 审计。

In [ ]:
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_logits=model(X,base_edges)  # 计算并保存当前步骤的中间状态。
    final_pred=final_logits.argmax(1)  # 计算并保存当前步骤的中间状态。
test_accuracy=float((final_pred[test_mask]==y[test_mask]).float().mean())  # 计算并保存当前步骤的中间状态。
original_train_loss=F.cross_entropy(final_logits[train_mask],y[train_mask])  # 计算并保存当前步骤的中间状态。
poisoned_y=y.clone(); poisoned_y[test_mask]=1-poisoned_y[test_mask]  # 计算并保存当前步骤的中间状态。
poisoned_train_loss=F.cross_entropy(final_logits[train_mask],poisoned_y[train_mask])  # 计算并保存当前步骤的中间状态。
assert 0<=test_accuracy<=1  # 用受控断言验证关键不变量。
assert torch.equal(y[train_mask],poisoned_y[train_mask])  # 用受控断言验证关键不变量。
assert torch.allclose(original_train_loss,poisoned_train_loss,atol=0,rtol=0)  # 用受控断言验证关键不变量。
assert not torch.equal(y[test_mask],poisoned_y[test_mask])  # 用受控断言验证关键不变量。

## 11. 查看 attention，但不要当作因果解释

下面仅列出第一层 head-0 对一个目标节点的最高入边权重。attention 是模型内部、特定层/head/参数化下的归一化系数；它不表示干预后的因果效应，也没有说明 source 特征的正负方向。生产展示还需权限过滤、稳定节点 ID、层/head 标识和模型版本。

In [ ]:
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _,final_attention=model(X,base_edges,return_attention=True)  # 计算并保存当前步骤的中间状态。
alpha1,edges1=final_attention["layer1"]  # 计算并保存当前步骤的中间状态。
target_node=0  # 计算并保存当前步骤的中间状态。
incoming=torch.where(edges1[1]==target_node)[0]  # 计算并保存当前步骤的中间状态。
ranked=sorted([(node_ids[int(edges1[0,i])],float(alpha1[i,0])) for i in incoming],key=lambda x:(-x[1],x[0]))  # 计算并保存当前步骤的中间状态。
attention_sum=sum(score for _,score in ranked)  # 计算并保存当前步骤的中间状态。
assert len(ranked)==int((edges1[1]==target_node).sum())  # 用受控断言验证关键不变量。
assert abs(attention_sum-1.0)<1e-6  # 用受控断言验证关键不变量。
assert ranked[0][1]>=ranked[-1][1]>=0  # 用受控断言验证关键不变量。
assert all(node.startswith("tenant-a:") for node,_ in ranked)  # 用受控断言验证关键不变量。

## 12. tenant 泄漏：softmax 分母也会泄漏

把 tenant-b 极端节点通过恶意边连到 node 0 后，全图 attention 的候选集合、分母和消息都会改变。下面用同一已训练模型对比安全图与未过滤原始图；差异证明“最后裁掉 tenant-b 输出”不够。

In [ ]:
raw_edges=directed_edges(20,raw_pairs)  # 计算并保存当前步骤的中间状态。
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    safe_node0=model(X,base_edges)[0]  # 计算并保存当前步骤的中间状态。
    unsafe_node0=model(X_all,raw_edges)[0]  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(safe_node0,unsafe_node0)  # 用受控断言验证关键不变量。
unsafe_with_loops=with_exact_self_loops(raw_edges,20)  # 计算并保存当前步骤的中间状态。
assert (18,0) in set(map(tuple,unsafe_with_loops.T.tolist()))  # 用受控断言验证关键不变量。
assert (18,0) not in set(map(tuple,masked_edges.T.tolist()))  # 用受控断言验证关键不变量。
assert X.shape[0]<X_all.shape[0]  # 用受控断言验证关键不变量。

## 13. Artifact 与图/特征/state 绑定

GAT 对 edge mask 和节点顺序敏感，因此制品必须绑定授权图指纹、feature schema、自环策略、head 结构及 state_dict。state 指纹按 key、dtype、shape、原始字节计算。内容哈希可发现错配，但生产仍应使用签名制品、不可变注册表和受控加载。

除 schema 外，制品还绑定 `snapshot_id/as_of/node_order/shape/dtype/content_sha256` 组成的有序特征快照。验证函数从当前实际 `X` 重建描述；任一特征值变化都会被拒绝。architecture 也记录 concat、negative slope 和 dropout，避免相同权重被不同 forward 语义解释。


In [ ]:
FEATURE_SCHEMA = {"order": ["http_ratio","batch_ratio","cpu_norm","bias"],  # 计算并保存当前步骤的中间状态。
                  "dtype": "float32", "source": "synthetic-v1", "fit_split": "not_applicable"}  # 执行当前语句以推进本节示例。
FEATURE_SCHEMA_ID = canonical_fingerprint(FEATURE_SCHEMA)  # 计算并保存当前步骤的中间状态。
FEATURE_SNAPSHOT_ID = "tenant-a-gat-features-2026-07-01T00:00:00Z"  # 计算并保存当前步骤的中间状态。
FEATURE_SNAPSHOT_AS_OF = "2026-07-01T00:00:00Z"  # 计算并保存当前步骤的中间状态。

def feature_snapshot_descriptor(ordered_node_ids, features, snapshot_id: str, as_of: str) -> dict:  # 定义本节可复用的核心函数。
    if features.ndim != 2 or len(ordered_node_ids) != features.shape[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("节点顺序与特征 shape 不匹配")  # 遇到非法合同立即显式失败。
    if len(set(ordered_node_ids)) != len(ordered_node_ids) or not torch.isfinite(features).all():  # 按当前条件选择后续控制路径。
        raise ValueError("节点 ID 必须唯一且特征必须有限")  # 遇到非法合同立即显式失败。
    value = features.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
    return {"snapshot_id": snapshot_id, "as_of": as_of,  # 返回当前分支计算出的结果。
            "node_order": list(ordered_node_ids), "shape": list(value.shape),  # 执行当前语句以推进本节示例。
            "dtype": str(value.dtype),  # 执行当前语句以推进本节示例。
            "content_sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest()}  # 执行当前语句以推进本节示例。

FEATURE_SNAPSHOT = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
    node_ids, X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
FEATURE_SNAPSHOT_FINGERPRINT = canonical_fingerprint(FEATURE_SNAPSHOT)  # 计算并保存当前步骤的中间状态。
graph_payload = {  # 计算并保存当前步骤的中间状态。
    "tenant": auth_a.tenant, "as_of": "2026-07-01T00:00:00Z", "nodes": node_ids,  # 执行当前语句以推进本节示例。
    "edges": sorted([sorted((node_ids[u],node_ids[v])) for u,v in undirected_pairs]),  # 执行当前语句以推进本节示例。
    "self_loop_policy": "exactly-one", "edge_direction": "source-to-target",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
GRAPH_FINGERPRINT = canonical_fingerprint(graph_payload)  # 计算并保存当前步骤的中间状态。

def state_dict_fingerprint(state) -> str:  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()[:20]  # 返回当前分支计算出的结果。

artifact = {  # 计算并保存当前步骤的中间状态。
    "model_type": "GATNet-from-scratch", "model_version": "gat-v2",  # 执行当前语句以推进本节示例。
    "tenant": auth_a.tenant, "as_of": graph_payload["as_of"],  # 执行当前语句以推进本节示例。
    "graph_fingerprint": GRAPH_FINGERPRINT, "feature_schema_id": FEATURE_SCHEMA_ID,  # 执行当前语句以推进本节示例。
    "feature_snapshot_id": FEATURE_SNAPSHOT_ID,  # 执行当前语句以推进本节示例。
    "feature_snapshot_fingerprint": FEATURE_SNAPSHOT_FINGERPRINT,  # 执行当前语句以推进本节示例。
    "architecture": {"dims": [4,4,2], "heads": [2,1], "concat": [True,False],  # 执行当前语句以推进本节示例。
                     "negative_slope": 0.2, "hidden_dropout": 0.1,  # 执行当前语句以推进本节示例。
                     "attention_dropout": [0.05,0.0]},  # 执行当前语句以推进本节示例。
    "self_loop_policy": "exactly-one",  # 执行当前语句以推进本节示例。
    "state_dict_fingerprint": state_dict_fingerprint(model.state_dict()),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact["artifact_id"] = canonical_fingerprint(artifact)  # 计算并保存当前步骤的中间状态。

def validate_artifact(value, state, current_feature_snapshot) -> bool:  # 定义本节可复用的核心函数。
    unsigned = {k:v for k,v in value.items() if k != "artifact_id"}  # 计算并保存当前步骤的中间状态。
    if canonical_fingerprint(unsigned) != value.get("artifact_id"):  # 按当前条件选择后续控制路径。
        raise ValueError("artifact hash 不匹配")  # 遇到非法合同立即显式失败。
    if (value.get("graph_fingerprint") != GRAPH_FINGERPRINT or  # 按当前条件选择后续控制路径。
            value.get("feature_schema_id") != FEATURE_SCHEMA_ID):  # 计算并保存当前步骤的中间状态。
        raise ValueError("图或特征 schema 不匹配")  # 遇到非法合同立即显式失败。
    current_feature_fp = canonical_fingerprint(current_feature_snapshot)  # 计算并保存当前步骤的中间状态。
    if (value.get("feature_snapshot_id") != current_feature_snapshot.get("snapshot_id") or  # 按当前条件选择后续控制路径。
            value.get("feature_snapshot_fingerprint") != current_feature_fp):  # 计算并保存当前步骤的中间状态。
        raise ValueError("有序特征快照不匹配")  # 遇到非法合同立即显式失败。
    if (value.get("self_loop_policy") != "exactly-one" or  # 按当前条件选择后续控制路径。
            value.get("state_dict_fingerprint") != state_dict_fingerprint(state)):  # 计算并保存当前步骤的中间状态。
        raise ValueError("自环或 state_dict 合同不匹配")  # 遇到非法合同立即显式失败。
    return True  # 返回当前分支计算出的结果。

assert validate_artifact(artifact, model.state_dict(), FEATURE_SNAPSHOT)  # 用受控断言验证关键不变量。
assert len(artifact["artifact_id"]) == 20  # 用受控断言验证关键不变量。
for field, bad in (("graph_fingerprint","bad"),  # 遍历输入元素以累积或检查结果。
                   ("feature_schema_id","bad"),  # 执行当前语句以推进本节示例。
                   ("feature_snapshot_fingerprint","bad"),  # 执行当前语句以推进本节示例。
                   ("state_dict_fingerprint","bad")):  # 执行当前语句以推进本节示例。
    forged = dict(artifact); forged[field] = bad  # 计算并保存当前步骤的中间状态。
    forged["artifact_id"] = canonical_fingerprint({k:v for k,v in forged.items() if k != "artifact_id"})  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        validate_artifact(forged, model.state_dict(), FEATURE_SNAPSHOT)  # 执行当前语句以推进本节示例。
        raise AssertionError(f"伪造 {field} 未拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。

changed_X27 = X.clone(); changed_X27[0, 0] += 25.0  # 计算并保存当前步骤的中间状态。
changed_feature_snapshot27 = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
    node_ids, changed_X27, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert canonical_fingerprint(changed_feature_snapshot27) != FEATURE_SNAPSHOT_FINGERPRINT  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(artifact, model.state_dict(), changed_feature_snapshot27)  # 执行当前语句以推进本节示例。
    raise AssertionError("特征内容改变后仍通过 artifact 校验")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "特征快照" in str(exc)  # 用受控断言验证关键不变量。


## 14. 推理合同

客户端只提交稳定节点 ID，不能提交 edge mask、覆盖 tenant 或指定另一个 graph fingerprint。服务从内部授权快照加载 `X/base_edges/model/artifact`，验证 bundle 后整图前向，再裁出请求节点。默认只返回概率和 trace；若开放 attention 诊断，必须单独 scope、限量并经过邻居 ACL。

服务从当前实际 `X` 重算有序特征快照并校验，trace 同时携带 snapshot ID、as-of 与内容指纹。由此，同一 graph fingerprint 下的特征漂移不能静默复用旧制品。


In [ ]:
def predict_known(auth: AuthContext, requested: list[str], model, artifact):  # 定义本节可复用的核心函数。
    auth.require("model:predict")  # 执行当前语句以推进本节示例。
    if auth.tenant != artifact.get("tenant"):  # 按当前条件选择后续控制路径。
        raise PermissionError("tenant 与制品不匹配")  # 遇到非法合同立即显式失败。
    current_feature_snapshot = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
        node_ids, X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    validate_artifact(artifact, model.state_dict(), current_feature_snapshot)  # 执行当前语句以推进本节示例。
    index = {node:i for i,node in enumerate(node_ids)}  # 计算并保存当前步骤的中间状态。
    if not requested or any(node not in index for node in requested):  # 按当前条件选择后续控制路径。
        raise PermissionError("未知或跨 tenant 节点")  # 遇到非法合同立即显式失败。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        probs = model(X, base_edges).softmax(1)  # 计算并保存当前步骤的中间状态。
    return {"predictions": [{"node_id":node, "probabilities":probs[index[node]].tolist()}  # 返回当前分支计算出的结果。
                            for node in requested],  # 遍历输入元素以累积或检查结果。
            "trace": {"artifact_id":artifact["artifact_id"],  # 执行当前语句以推进本节示例。
                      "graph_fingerprint":GRAPH_FINGERPRINT,  # 执行当前语句以推进本节示例。
                      "feature_schema_id":FEATURE_SCHEMA_ID,  # 执行当前语句以推进本节示例。
                      "feature_snapshot_id":current_feature_snapshot["snapshot_id"],  # 执行当前语句以推进本节示例。
                      "feature_snapshot_as_of":current_feature_snapshot["as_of"],  # 执行当前语句以推进本节示例。
                      "feature_snapshot_fingerprint":canonical_fingerprint(current_feature_snapshot),  # 执行当前语句以推进本节示例。
                      "attention_exposed":False}}  # 执行当前语句以推进本节示例。

served = predict_known(auth_a, node_ids[:2], model, artifact)  # 计算并保存当前步骤的中间状态。
assert len(served["predictions"]) == 2  # 用受控断言验证关键不变量。
assert served["trace"]["artifact_id"] == artifact["artifact_id"]  # 用受控断言验证关键不变量。
assert served["trace"]["feature_snapshot_fingerprint"] == FEATURE_SNAPSHOT_FINGERPRINT  # 用受控断言验证关键不变量。
assert all(abs(sum(row["probabilities"])-1) < 1e-6 for row in served["predictions"])  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    predict_known(auth_a, ["tenant-b:x0"], model, artifact)  # 执行当前语句以推进本节示例。
    raise AssertionError("越权节点未拒绝")  # 遇到非法合同立即显式失败。
except PermissionError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 15. 复杂度、数值与生产替换点

本实现每层投影约 (O(NHF_{in}F_{out}))，边打分/聚合约 (O(EHF_{out}))。Python 按节点循环 segment softmax 仅适合公式教学；生产应使用融合 scatter/segment kernel、稀疏批处理、邻居采样、混合精度下的稳定 softmax、显存预算和超时降级。不要构造 (N\times N) 全连接 attention mask。

上线需报告 0-hop/GCN/GraphSAGE 基线、多 seed 置信区间、head/度数/时间切片、attention 熵与饱和、标签延迟、动态图版本、tenant 越权、模型签名、灰度与回滚。attention 可视化不是因果解释，受控小图高分不能外推。

In [ ]:
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _, checked_attention = model(X, base_edges, return_attention=True)  # 计算并保存当前步骤的中间状态。
for alpha, edges in checked_attention.values():  # 遍历输入元素以累积或检查结果。
    for node in range(18):  # 遍历输入元素以累积或检查结果。
        assert torch.allclose(alpha[edges[1] == node].sum(0), torch.ones(alpha.shape[1]), atol=1e-6)  # 用受控断言验证关键不变量。
assert isinstance(model.gat1, GATLayer) and isinstance(model.gat2, GATLayer)  # 用受控断言验证关键不变量。
assert model.gat1.attn_source.grad is not None and model.gat2.attn_target.grad is not None  # 用受控断言验证关键不变量。
assert state_dict_fingerprint(model.state_dict()) == artifact["state_dict_fingerprint"]  # 用受控断言验证关键不变量。
assert GRAPH_FINGERPRINT == canonical_fingerprint(graph_payload)  # 用受控断言验证关键不变量。
assert updated_parameter_keys and restored_train_eval_loss < initial_train_eval_loss  # 用受控断言验证关键不变量。
assert artifact["feature_snapshot_fingerprint"] == canonical_fingerprint(FEATURE_SNAPSHOT)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_out27, torch.tensor([[4.0],[12.0]]), atol=1e-7)  # 用受控断言验证关键不变量。
assert train_seconds < 20 and validate_artifact(artifact, model.state_dict(), FEATURE_SNAPSHOT)  # 用受控断言验证关键不变量。
print({"status":"PASS", "model":"GAT-from-scratch", "best_epoch":best_epoch,  # 执行当前语句以推进本节示例。
       "test_accuracy":round(test_accuracy,3),  # 执行当前语句以推进本节示例。
       "train_loss_before_after":[round(initial_train_eval_loss,4),round(restored_train_eval_loss,4)],  # 执行当前语句以推进本节示例。
       "params":sum(p.numel() for p in model.parameters()), "seconds":round(train_seconds,3)})  # 执行当前语句以推进本节示例。


## 16. 原始与官方资料

- Veličković et al., *Graph Attention Networks*：https://arxiv.org/abs/1710.10903
- PyTorch 官方 `nn.Module` 文档：https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch 官方 Module/State 文档：https://docs.pytorch.org/docs/stable/notes/modules.html
- PyTorch 官方初始化 API：https://docs.pytorch.org/docs/stable/nn.init.html

原论文用于 masked self-attention、多头与 concat/average 背景；目标分段实现、tenant 前置过滤、制品指纹、注意力暴露策略和生产失败边界是本 Notebook 的工程扩展。